# الفصل 14: الترميز الموضعي المتقدم وآليات الانتباه الفعّالة

> هذا الفصل هو "صندوق أدوات" الهندسة المعمارية الأكثر تأثيرًا في نماذج اللغة الحديثة بين 2021 و2024.

- **الجزء:** 4 — المعماريات الحديثة
- **الفصل:** 14
- **المستوى:** متقدم
- **زمن القراءة:** 30 دقيقة
- **المتطلبات:** الفصل 13: حدود المحوّل الكلاسيكي
- **الوسوم:** RoPE، ALiBi، GQA، Flash Attention

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch14-modern-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch14-modern-attention.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch14-modern-attention.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> هذا الفصل هو "صندوق أدوات" الهندسة المعمارية الأكثر تأثيرًا في نماذج اللغة الحديثة بين 2021 و2024.

> **بيانات الفصل**
>
> - **الجزء:** 4 — المعماريات الحديثة
> - **المستوى:** متقدم
> - **زمن القراءة:** 30 دقيقة
> - **المتطلبات:** الفصل 13: حدود المحوّل الكلاسيكي


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم ALiBi بالتفصيل
- فهم MQA وGQA بالتفصيل
- فهم Flash Attention بالتفصيل


## 1. لماذا يهمّك هذا الموضوع؟

هذا الفصل هو "صندوق أدوات" الهندسة المعمارية الأكثر تأثيرًا في نماذج اللغة الحديثة بين 2021 و2024. ستتعلم هنا الابتكارات التي تجعل LLaMA وMistral وGPT-4 وClaude قادرة على معالجة سياقات بعشرات أو مئات آلاف الرموز بكفاءة: **RoPE** (الترميز الموضعي الدوّار، المعيار شبه الحصري اليوم)، **ALiBi** (بديل أبسط عبر تحيّز خطي)، **MQA وGQA** (لتقليل حجم KV Cache الذي ناقشناه في الفصل السابق)، **Flash Attention** (تسريع هائل دون تغيير النتيجة الرياضية)، و**Ring Attention** (لتوزيع الحساب على عدة معالجات لسياقات تتجاوز سعة جهاز واحد).

---


## 2. الخريطة الذهنية


```text
المشكلة 1: الترميز الموضعي التقليدي لا يستقرئ جيدًا لأطوال أطول
│
├── RoPE (Rotary Positional Embedding)
│   → تدوير متجهات Q,K بزاوية تعتمد على الموضع النسبي
│   → يُطبَّق مباشرة داخل عملية الانتباه (ليس إضافة للتضمين)
│   → يحافظ على معلومات الموضع النسبي بشكل طبيعي جدًا
│   → المعيار الفعلي في LLaMA، Mistral، Qwen، DeepSeek
│
└── ALiBi (Attention with Linear Biases)
    → لا يُعدِّل Q,K إطلاقًا؛ بل يضيف تحيزًا سلبيًا خطيًا لدرجات الانتباه
      يتناسب مع المسافة بين الموضعين
    → بسيط جدًا للتنفيذ، استقراء ممتاز لأطوال أطول بكثير من التدريب
    → مستخدَم في BLOOM وبعض النماذج الأخرى

المشكلة 2: KV Cache ضخم جدًا مع عدد كبير من رؤوس الانتباه
│
├── MHA (Multi-Head Attention) الأصلي:
│   كل رأس له Q, K, V خاصة به بالكامل → KV Cache كامل لكل رأس
│
├── MQA (Multi-Query Attention):
│   كل الرؤوس تتشارك K وV واحدة فقط (Q منفصلة لكل رأس)
│   → تقليل KV Cache بعامل num_heads! لكن فقدان بعض الجودة
│
└── GQA (Grouped-Query Attention):
    تقسيم الرؤوس لمجموعات، كل مجموعة تتشارك K,V واحدة
    → توازن ممتاز بين كفاءة MQA وجودة MHA
    → المعيار الفعلي في LLaMA-2/3، Mistral، معظم LLMs الحديثة

المشكلة 3: تنفيذ Attention القياسي يستهلك ذاكرة زائدة
│
└── Flash Attention:
    إعادة تنظيم الحساب (Tiling + Online Softmax) لتجنب تخزين
    مصفوفة n×n الكاملة في الذاكرة البطيئة (HBM)
    → نفس النتيجة الرياضية بالضبط، أسرع وأخف ذاكريًا بشكل كبير

المشكلة 4: سياق أطول من أن يُعالَج على GPU واحد
│
└── Ring Attention:
    توزيع أجزاء التسلسل عبر عدة GPUs مرتبة في "حلقة"،
    مع تمرير أجزاء K,V دوريًا بين الأجهزة أثناء الحساب
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

**RoPE**: تخيّل عقارب ساعة تدور بزاوية معينة حسب "موضعها الزمني". بدلاً من إضافة رقم ثابت يمثل الموضع (كما في PE التقليدي)، فإن RoPE "يُدوِّر" متجه كل كلمة بزاوية تتناسب مع موضعها، بحيث عندما تُضرَب متجهات كلمتين (في عملية الانتباه)، فإن الفرق بين زاويتي الدوران يُصبح تلقائيًا معبّرًا عن **المسافة النسبية** بينهما — بطريقة أنيقة رياضيًا جدًا.

**ALiBi**: فكرة أبسط من ذلك بكثير: لا تُعدِّل التمثيلات إطلاقًا، فقط "عاقب" درجات الانتباه بمقدار يتناسب مع المسافة — كلما بعدت كلمتان، كلما طُرح رقم أكبر من درجة توافقهما، مما يجعل النموذج "يميل طبيعيًا" للانتباه للكلمات القريبة أكثر، مع إبقاء إمكانية الانتباه للبعيدة عند الحاجة الفعلية.

**MQA/GQA**: تخيّل مكتبة فيها 32 قارئًا (الرؤوس)، كل قارئ يحتاج نسخته الخاصة من الكتاب (K,V). MHA يُعطي كل قارئ نسخته الخاصة بالكامل (مُكلف تخزينًا). MQA يُعطي كل القراء نسخة واحدة مشتركة من الكتاب (أرخص لكن قد يُقيّد التنوع). GQA هو الحل الوسط: نُقسّم القراء لمجموعات صغيرة، وكل مجموعة تتشارك نسخة واحدة.

**Flash Attention**: تخيّل أنك تطبخ وجبة كبيرة في مطبخ صغير المساحة: بدلاً من إخراج كل المكونات دفعة واحدة على الطاولة (يملأ المساحة بسرعة)، تُحضّر وتُعالج المكونات على دفعات صغيرة (tiles) وتُراكم النتيجة النهائية تدريجيًا — تحصل على نفس الطبق بالضبط، لكن بمساحة عمل أقل بكثير.

### 🟡 طبقة الآلية: RoPE بالتفصيل الرياضي

الفكرة الأساسية لـ RoPE: بدلاً من إضافة ترميز موضعي للتضمين (كما في PE الجيبي)، نُطبّق **دورانًا (rotation)** على متجهي Query وKey أنفسهما، بزاوية تتناسب مع الموضع.

لمتجه ثنائي الأبعاد (أبسط حالة توضيحية) عند الموضع $m$:

$$f(x, m) = \begin{pmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}$$

حيث $\theta$ تردد ثابت. لمتجه عام بأبعاد $d$ (زوجية)، نُقسّمه إلى $d/2$ أزواج، ونُطبّق دورانًا مستقلًا على كل زوج بتردد مختلف $\theta_i = 10000^{-2i/d}$ (نفس فكرة الترددات المتعددة في PE الجيبي الأصلي، لكن مُطبَّقة كدوران وليس إضافة).

**الخاصية السحرية**: عندما نحسب حاصل الضرب النقطي بين Query عند الموضع $m$ وKey عند الموضع $n$ بعد الدوران:

$$\langle f(q, m), f(k, n) \rangle = g(q, k, m-n)$$

أي أن النتيجة تعتمد **فقط على الفرق $m-n$** (المسافة النسبية) وليس على الموضعين المطلقين بشكل منفصل! هذه الخاصية الرياضية الأنيقة تعني أن النموذج يتعلم تلقائيًا علاقات تعتمد على المسافة النسبية بين الكلمات، وهي الخاصية الأكثر فائدة لغويًا (العلاقة بين كلمتين متجاورتين يجب أن تكون متشابهة بغض النظر عن كونهما في بداية الجملة أو نهايتها).

**لماذا يستقرئ RoPE بشكل أفضل لسياقات أطول؟** لأن الدوران عملية **مستمرة ودورية**، يمكن تمديدها رياضيًا لمواضع لم يرها النموذج أثناء التدريب، على عكس Learned PE المحدود بعدد صفوف ثابت في جدول البحث (lookup table).

### 🟡 ALiBi بالتفصيل

بدلاً من أي تعديل على Q أو K، يُضاف تحيّز ثابت (غير قابل للتعلم) مباشرة لدرجات الانتباه قبل softmax:

$$\text{scores}_{i,j} = Q_iK_j^T - m \cdot |i - j|$$

حيث $m$ "ميل" (slope) ثابت يختلف لكل رأس من رؤوس الانتباه (رؤوس مختلفة تحصل على ميول مختلفة، محسوبة كمتتالية هندسية). كلما ابتعد الموضعان $i$ و$j$، زادت العقوبة المطروحة، مما يُقلِّل تلقائيًا أهمية الكلمات البعيدة جدًا دون حذفها كليًا.

### 🟡 MQA وGQA بالتفصيل

في **MHA القياسي**: لكل رأس $i$ من $h$ رؤوس، لدينا $W_i^Q, W_i^K, W_i^V$ منفصلة تمامًا.

في **MQA**: لدينا $W_i^Q$ منفصلة لكل رأس، لكن $W^K$ و$W^V$ **مشتركتان** بين كل الرؤوس — أي أن حجم ذاكرة KV Cache يصبح بمقدار $1/h$ من حجمه في MHA.

في **GQA**: نُقسّم $h$ رأسًا إلى $g$ مجموعات ($g < h$)، كل مجموعة من الرؤوس تتشارك K,V خاصة بها (مختلفة عن المجموعات الأخرى). هذا يُعطي تحكمًا دقيقًا بالتوازن: $g=1$ يُعادل MQA تمامًا، و$g=h$ يُعادل MHA تمامًا.

### 🟡 Flash Attention بالتفصيل

المشكلة التي يحلّها Flash Attention ليست خوارزمية (النتيجة الرياضية مطابقة تمامًا لـ Attention القياسي) بل **هندسية/حسابية**: في GPU، هناك طبقتان من الذاكرة — **HBM** (ذاكرة كبيرة لكن بطيئة نسبيًا، عشرات الجيجابايتات) و**SRAM** (ذاكرة صغيرة جدًا لكن سريعة جدًا، ميجابايتات قليلة داخل كل نواة معالجة). التنفيذ القياسي لـ Attention يكتب مصفوفة $n \times n$ الكاملة إلى HBM البطيئة عدة مرات (عند حساب $QK^T$، ثم softmax، ثم الضرب بـ V) — وهذا النقل المتكرر بين الذاكرتين هو عنق الزجاجة الفعلي، وليس العمليات الحسابية نفسها.

**حل Flash Attention**: تقسيم Q وK وV إلى كتل صغيرة (tiles) تتسع في SRAM السريعة، وحساب الانتباه بشكل تدريجي (**online softmax** — خوارزمية ذكية تُحدِّث تقديرات softmax تدريجيًا دون الحاجة لرؤية كل الصف دفعة واحدة)، مع تجميع النتيجة النهائية دون تخزين مصفوفة $n \times n$ الكاملة في HBM إطلاقًا. النتيجة: تسريع يصل لـ 2-4 أضعاف وتقليل ذاكرة كبير جدًا، **بنفس الناتج الرياضي بالضبط** (ليس تقريبًا).

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import math

# ============================================================
# 1. RoPE (Rotary Positional Embedding) — التطبيق الكامل
# ============================================================
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 8192, base: float = 10000.0):
        super().__init__()
        # ترددات مختلفة لكل زوج من الأبعاد: θ_i = base^(-2i/dim)
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=torch.float)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)  # (seq_len, dim/2)
        # نُكرّر كل تردد مرتين لتغطية الزوج الكامل من الأبعاد
        emb = torch.cat([freqs, freqs], dim=-1)             # (seq_len, dim)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)

    def forward(self, seq_len: int):
        if seq_len > self.max_seq_len:
            self._build_cache(seq_len)
            self.max_seq_len = seq_len
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x):
    """تدوير نصفي: [x1, x2] → [-x2, x1]، خطوة أساسية في تطبيق RoPE."""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    تطبيق الدوران على Q وK.
    q, k: (batch, num_heads, seq_len, head_dim)
    cos, sin: (seq_len, head_dim)
    """
    cos = cos.unsqueeze(0).unsqueeze(0)  # للبث على batch وnum_heads
    sin = sin.unsqueeze(0).unsqueeze(0)

    q_rotated = (q * cos) + (rotate_half(q) * sin)
    k_rotated = (k * cos) + (rotate_half(k) * sin)
    return q_rotated, k_rotated

class MultiHeadAttentionWithRoPE(nn.Module):
    """Multi-Head Attention مع RoPE بدلاً من الترميز الموضعي الإضافي."""
    def __init__(self, d_model, num_heads, max_seq_len=8192, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.rope = RotaryPositionalEmbedding(self.d_k, max_seq_len)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # تطبيق RoPE على Q وK فقط (ليس V)
        cos, sin = self.rope(seq_len)
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-1e9'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_o(context)

# ============================================================
# 2. ALiBi (Attention with Linear Biases)
# ============================================================
def get_alibi_slopes(num_heads: int) -> torch.Tensor:
    """
    حساب الميول (slopes) لكل رأس، كمتتالية هندسية كما في الورقة الأصلية.
    """
    def get_slopes_power_of_2(n):
        start = 2 ** (-2 ** -(math.log2(n) - 3))
        ratio = start
        return [start * ratio ** i for i in range(n)]

    if math.log2(num_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(num_heads))
    else:
        # معالجة الحالات غير القوى الصحيحة لـ 2 (تفصيل من الورقة الأصلية)
        closest_power_of_2 = 2 ** math.floor(math.log2(num_heads))
        slopes = get_slopes_power_of_2(closest_power_of_2)
        extra = get_slopes_power_of_2(2 * closest_power_of_2)[0::2][:num_heads - closest_power_of_2]
        return torch.tensor(slopes + extra)

class MultiHeadAttentionWithALiBi(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        slopes = get_alibi_slopes(num_heads)
        self.register_buffer('slopes', slopes)
        self.dropout = nn.Dropout(dropout)

    def _get_alibi_bias(self, seq_len, device):
        # مصفوفة المسافات بين كل زوج مواضع: (seq_len, seq_len)
        positions = torch.arange(seq_len, device=device)
        relative_positions = positions.unsqueeze(0) - positions.unsqueeze(1)
        # نريد فقط المسافات غير الموجبة (causal) لتعكس البعد للخلف
        relative_positions = -relative_positions.abs().float()

        # لكل رأس: ميل خاص × مصفوفة المسافات
        alibi_bias = self.slopes.view(-1, 1, 1) * relative_positions.unsqueeze(0)
        return alibi_bias  # (num_heads, seq_len, seq_len)

    def forward(self, x, causal_mask=None):
        batch_size, seq_len, _ = x.shape

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # إضافة تحيز ALiBi (بدلًا من أي ترميز موضعي على Q,K)
        alibi_bias = self._get_alibi_bias(seq_len, x.device).unsqueeze(0)
        scores = scores + alibi_bias

        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask == 0, float('-1e9'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_o(context)

# ============================================================
# 3. Grouped-Query Attention (GQA) — يشمل MQA كحالة خاصة
# ============================================================
class GroupedQueryAttention(nn.Module):
    """
    GQA: تقسيم رؤوس Query إلى مجموعات، كل مجموعة تتشارك K,V.
    num_kv_groups=1  → مكافئ MQA
    num_kv_groups=num_heads → مكافئ MHA القياسي
    """
    def __init__(self, d_model, num_heads, num_kv_groups, max_seq_len=8192, dropout=0.1):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads يجب أن يُقسَم على num_kv_groups"

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.d_k = d_model // num_heads
        self.group_size = num_heads // num_kv_groups  # عدد رؤوس Q لكل مجموعة K,V

        self.W_q = nn.Linear(d_model, num_heads * self.d_k, bias=False)
        # ملاحظة: K,V بأبعاد أصغر بكثير — فقط num_kv_groups بدلًا من num_heads
        self.W_k = nn.Linear(d_model, num_kv_groups * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_groups * self.d_k, bias=False)
        self.W_o = nn.Linear(num_heads * self.d_k, d_model, bias=False)

        self.rope = RotaryPositionalEmbedding(self.d_k, max_seq_len)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_kv_groups, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_kv_groups, self.d_k).transpose(1, 2)

        cos, sin = self.rope(seq_len)
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)

        # تكرار K,V لتُطابق عدد رؤوس Q (كل مجموعة K,V تُستخدَم لعدة رؤوس Q)
        K = K.repeat_interleave(self.group_size, dim=1)  # (batch, num_heads, seq_len, d_k)
        V = V.repeat_interleave(self.group_size, dim=1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-1e9'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        return self.W_o(context)

def compute_kv_cache_size(num_layers, num_kv_heads, head_dim, seq_len,
                            batch_size=1, bytes_per_element=2):
    """حساب حجم KV Cache بالجيجابايت لمقارنة MHA/GQA/MQA."""
    # العامل 2: K و V منفصلان
    total_bytes = (2 * num_layers * num_kv_heads * head_dim *
                   seq_len * batch_size * bytes_per_element)
    return total_bytes / (1024 ** 3)  # تحويل لجيجابايت

# ============================================================
# 4. Flash Attention — استخدام التطبيق الرسمي المُحسَّن (PyTorch 2.0+)
# ============================================================
def flash_attention_demo(Q, K, V, is_causal=True):
    """
    PyTorch 2.0+ يوفر scaled_dot_product_attention المُحسَّنة
    التي تستخدم Flash Attention تلقائيًا عند توفر العتاد المناسب.
    """
    # هذه الدالة تختار تلقائيًا أسرع تطبيق متاح (Flash/Memory-Efficient/Math)
    output = torch.nn.functional.scaled_dot_product_attention(
        Q, K, V, is_causal=is_causal, dropout_p=0.0
    )
    return output

# مقارنة عملية: التنفيذ اليدوي مقابل Flash Attention المدمج
def compare_standard_vs_flash(seq_len=4096, d_model=512, num_heads=8, device='cuda'):
    import time

    batch_size = 2
    d_k = d_model // num_heads
    Q = torch.randn(batch_size, num_heads, seq_len, d_k, device=device, dtype=torch.float16)
    K = torch.randn(batch_size, num_heads, seq_len, d_k, device=device, dtype=torch.float16)
    V = torch.randn(batch_size, num_heads, seq_len, d_k, device=device, dtype=torch.float16)

    # التنفيذ القياسي (يدوي، يخزّن مصفوفة n×n كاملة)
    def standard_attention(Q, K, V):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
        scores = scores.masked_fill(~causal_mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return torch.matmul(weights, V)

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    out_standard = standard_attention(Q, K, V)
    torch.cuda.synchronize()
    standard_time = time.time() - start
    standard_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)

    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    out_flash = flash_attention_demo(Q, K, V, is_causal=True)
    torch.cuda.synchronize()
    flash_time = time.time() - start
    flash_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)

    print(f"Standard: {standard_time*1000:.2f}ms, {standard_memory:.2f}MB")
    print(f"Flash:    {flash_time*1000:.2f}ms, {flash_memory:.2f}MB")
    print(f"تسريع: {standard_time/flash_time:.2f}x | توفير ذاكرة: {standard_memory/flash_memory:.2f}x")

# ============================================================
# 5. مفهوم Ring Attention المبسّط (توزيع عبر عدة أجهزة)
# ============================================================
def ring_attention_concept_demo(num_devices: int, total_seq_len: int, d_model: int):
    """
    توضيح مفاهيمي مبسّط لفكرة Ring Attention (التنفيذ الفعلي
    يتطلب أدوات توزيع مثل torch.distributed وNCCL):

    1. يُقسَّم التسلسل الطويل إلى num_devices أجزاء متساوية
    2. كل جهاز يحتفظ بجزئه من Q محليًا طوال الوقت
    3. أجزاء K,V تدور بين الأجهزة في "حلقة" (ring) عبر num_devices خطوة
    4. في كل خطوة دوران، يحسب كل جهاز انتباه جزئي بين Q المحلي
       وK,V الحالي القادم من الجهاز المجاور، ويُراكم النتيجة
    5. بعد اكتمال الدورة الكاملة، كل جهاز يملك نتيجة انتباه كاملة
       ومصححة لجزئه من Q مقابل كل أجزاء K,V من كل الأجهزة
    """
    chunk_size = total_seq_len // num_devices
    print(f"عدد الأجهزة: {num_devices}")
    print(f"طول كل جزء محلي: {chunk_size} رمز")
    print(f"الذاكرة المطلوبة لكل جهاز لتخزين Q,K,V المحلية فقط: "
          f"~{3 * chunk_size * d_model * 2 / (1024**2):.2f} MB (بدلاً من التسلسل الكامل)")
    print("كل جهاز يحسب تدريجيًا، عبر دورات تبادل، انتباهًا كاملًا دون "
          "الحاجة لتخزين التسلسل الكامل في ذاكرة جهاز واحد")


---


## 4. مثال محلول خطوة بخطوة

**مقارنة حجم KV Cache بين MHA وGQA وMQA** لنموذج بالمواصفات التالية:
- 32 طبقة، 32 رأس انتباه، $d_k = 128$، سياق 8,000 رمز، دفعة = 1، دقة FP16 (بايتان لكل عنصر).

**MHA القياسي** (عدد مجموعات KV = 32، أي مطابق لعدد الرؤوس):

$$\text{الحجم} = 2 \times 32 \times 32 \times 128 \times 8000 \times 1 \times 2 \text{ بايت}$$
$$= 2 \times 32 \times 32 \times 128 \times 8000 \times 2 = 4{,}194{,}304{,}000 \text{ بايت} \approx 3.91 \text{ GB}$$

**GQA بـ 8 مجموعات**:

$$\text{الحجم} = 2 \times 32 \times 8 \times 128 \times 8000 \times 1 \times 2 \text{ بايت} \approx 0.98 \text{ GB}$$

**MQA (مجموعة واحدة فقط)**:

$$\text{الحجم} = 2 \times 32 \times 1 \times 128 \times 8000 \times 1 \times 2 \text{ بايت} \approx 0.12 \text{ GB}$$

**الخلاصة الكمية**: الانتقال من MHA إلى GQA بـ8 مجموعات وفّر **75% من ذاكرة KV Cache** (من 3.91GB إلى 0.98GB)، بينما MQA وفّر **97% تقريبًا** (إلى 0.12GB فقط) — لكن بثمن أكبر في جودة النموذج (أظهرت تجارب LLaMA-2 أن GQA يحقق توازنًا شبه مثالي: توفير كبير في الذاكرة مع فقدان ضئيل جدًا في الجودة مقارنة بـ MQA الأكثر عدوانية).

---


## 5. نصائح وحيل

> 💡 **استخدم `torch.nn.functional.scaled_dot_product_attention` دائمًا في PyTorch 2.0+**: هذه الدالة تختار تلقائيًا أفضل تطبيق متاح (Flash Attention 2، Memory-Efficient Attention، أو التنفيذ الرياضي القياسي) حسب العتاد والمدخلات، دون الحاجة لكتابة Flash Attention يدويًا.

> 💡 **RoPE + GQA هو المعيار الفعلي للنماذج الحديثة**: إذا كنت تبني نموذجًا جديدًا من الصفر اليوم، ابدأ بهذا المزيج (كما في LLaMA-2/3، Mistral، Qwen) إلا إذا كان لديك سبب قوي للخروج عنه.

> 💡 **اختر عدد مجموعات GQA بعناية**: عدد صغير جدًا من المجموعات يُقارب MQA (توفير أكبر، جودة أقل)؛ عدد كبير يُقارب MHA (جودة أعلى، توفير أقل) — القيمة الشائعة في LLaMA-2-70B هي 8 مجموعات لـ 64 رأس (نسبة 8:1).

> 💡 **ALiBi خيار ممتاز للبساطة والاستقراء الفائق**: إذا كانت أولويتك هي تبسيط التنفيذ والحصول على استقراء جيد جدًا لأطوال أبعد بكثير من التدريب (كما أظهرت ورقة BLOOM)، فكّر في ALiBi كبديل أبسط من RoPE.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **تطبيق RoPE على V بالخطأ**: RoPE يُطبَّق فقط على Q وK (لأن الهدف هو جعل درجة التوافق بينهما تعتمد على الموضع النسبي) — تطبيقه على V خطأ مفاهيمي شائع يُفسد الترميز.

> ⚠️ **نسيان `repeat_interleave` الصحيح في GQA**: يجب التأكد من أن ترتيب تكرار رؤوس K,V يتطابق تمامًا مع ترتيب تقسيم رؤوس Q إلى مجموعات، وإلا فإن كل رأس Q سينتبه لمجموعة K,V خاطئة.

> ⚠️ **الاعتقاد بأن Flash Attention يُغيّر النتيجة الرياضية**: Flash Attention **ليست** تقنية تقريبية (approximation) — إنها إعادة تنظيم للحساب تُنتج **نفس** النتيجة الرياضية تمامًا (في حدود الدقة العددية) لكن بكفاءة أعلى بكثير؛ الخلط بينها وبين تقنيات التقريب (مثل sparse attention) خطأ مفاهيمي شائع.

> ⚠️ **محاولة استخدام max_seq_len أصغر من الطول الفعلي للتسلسل في RoPE بدون إعادة بناء الكاش**: يجب التأكد من أن `_build_cache` يُعاد استدعاؤه (أو يُبنى بطول كافٍ منذ البداية) عند التعامل مع تسلسلات أطول من المتوقع مبدئيًا.

---


## 7. جدول مقارنة

### RoPE مقابل ALiBi مقابل الترميز الموضعي التقليدي

| الخاصية | Sinusoidal/Learned PE | RoPE | ALiBi |
|---|---|---|---|
| **آلية العمل** | إضافة للتضمين | دوران على Q,K | تحيز خطي على درجات الانتباه |
| **الاستقراء لأطوال أطول** | ضعيف | جيد جدًا | ممتاز |
| **التعقيد الحسابي الإضافي** | ضئيل جدًا | ضئيل | ضئيل جدًا |
| **المرونة مع المسافة النسبية** | غير مباشرة | طبيعية ومدمجة رياضيًا | طبيعية ومباشرة |
| **الاستخدام في نماذج 2023+** | نادر | الأكثر شيوعًا | شائع (BLOOM وغيره) |

### MHA مقابل MQA مقابل GQA

| الخاصية | MHA | GQA | MQA |
|---|---|---|---|
| **عدد مجموعات K,V** | = عدد الرؤوس | بين 1 وعدد الرؤوس | 1 فقط |
| **حجم KV Cache** | الأكبر | متوسط (قابل للضبط) | الأصغر |
| **جودة النموذج** | الأعلى | قريبة جدًا من MHA | أقل قليلًا |
| **سرعة الاستدلال** | الأبطأ | سريعة | الأسرع |
| **الاستخدام الفعلي** | نماذج أقدم/أصغر | LLaMA-2/3، Mistral | PaLM، بعض النماذج المبكرة |

---


## 8. ملخص أفضل الممارسات

- ✅ استخدم RoPE كخيار افتراضي للترميز الموضعي في أي نموذج Decoder-only جديد.
- ✅ استخدم GQA بدلاً من MHA الكامل لأي نموذج بحجم إنتاجي يُراد نشره بكفاءة.
- ✅ اعتمد `scaled_dot_product_attention` المدمجة في PyTorch بدلاً من إعادة تنفيذ Flash Attention يدويًا إلا لأغراض تعليمية بحتة.
- ✅ احسب دائمًا حجم KV Cache المتوقع قبل اختيار عدد مجموعات GQA المناسب لميزانيتك الذاكرية.
- ✅ فكّر في Ring Attention أو تقنيات توزيع مشابهة فقط عند الحاجة الفعلية لسياقات تتجاوز سعة ذاكرة جهاز GPU واحد.

---


## 9. أسئلة مقابلات

**س1: اشرح كيف يضمن RoPE أن درجة التوافق بين Query وKey تعتمد على المسافة النسبية فقط، وليس المواضع المطلقة.**

**الإجابة النموذجية**: RoPE يُطبّق دورانًا هندسيًا على متجهي Query وKey، بزاوية تتناسب خطيًا مع موضع كل منهما في التسلسل (الموضع m للـ Query، والموضع n للـ Key). الخاصية الرياضية الجوهرية لعمليات الدوران هي أن حاصل الضرب الداخلي (dot product) بين متجهين مُدارين بزاويتين مختلفتين يعتمد فقط على **الفرق بين الزاويتين**، وليس على كل زاوية بمفردها. بما أن الزاوية تتناسب مع الموضع، فإن الفرق بين الزاويتين يتناسب مع الفرق بين الموضعين (m-n، أي المسافة النسبية). لذلك، عندما نحسب $Q_m \cdot K_n$ بعد تطبيق RoPE، تكون النتيجة دالة في $(m-n)$ فقط، بغض النظر عن كون الكلمتين في بداية جملة قصيرة أو في منتصف مستند طويل جدًا — وهذا يمنح النموذج تعميمًا طبيعيًا لمفهوم "المسافة بين الكلمات" يصمد أمام تغيّر طول السياق الكلي.

**س2: ما الفرق الجوهري بين MQA وGQA، وأين تقع ميزة GQA تحديدًا؟**

**الإجابة النموذجية**: في MQA (Multi-Query Attention)، تحتفظ كل رؤوس الانتباه الـ h بمتجهات Query منفصلة خاصة بها، لكنها جميعًا **تتشارك** زوجًا واحدًا فقط من مصفوفتي Key وValue — هذا يُقلِّل حجم KV Cache بعامل h (عدد الرؤوس)، لكنه قد يُقلِّل أيضًا من قدرة النموذج على التقاط أنواع متعددة ومتمايزة من العلاقات، لأن كل الرؤوس "تنظر" لنفس تمثيل المحتوى (K,V) رغم اختلاف استعلاماتها. GQA (Grouped-Query Attention) هو حل وسط ذكي: يُقسِّم الرؤوس الـ h إلى g مجموعات أصغر (حيث 1 < g < h)، وكل مجموعة من الرؤوس تتشارك K,V خاصة بمجموعتها فقط (مختلفة عن المجموعات الأخرى). هذا يحافظ على قدر معقول من التنوع بين المجموعات المختلفة (على عكس MQA التي توحّد كل شيء في مجموعة واحدة)، بينما لا يزال يُقلِّل حجم KV Cache بشكل كبير جدًا (بعامل h/g بدلًا من h الكامل). التجارب العملية (مثل ورقة LLaMA-2) أظهرت أن GQA بعدد مجموعات معتدل (مثل 8 من أصل 64 رأس) يحقق جودة قريبة جدًا من MHA الكامل مع توفير ذاكري كبير جدًا، مما يجعله التوازن الأمثل عمليًا.

**س3: لماذا يُعتبر Flash Attention "تسريعًا هندسيًا" وليس "تقنية تقريبية"، وما الفرق الجوهري بين النوعين؟**

**الإجابة النموذجية**: التقنيات التقريبية (approximation techniques) لـ Attention، مثل Sparse Attention أو Linear Attention، تُغيِّر فعليًا **النتيجة الرياضية** المحسوبة — إما بحساب انتباه جزئي فقط بين مجموعة فرعية من المواضع (sparse)، أو باستبدال softmax بدالة تقريبية ذات تعقيد أقل (linear). هذا يعني أن هذه التقنيات قد تفقد بعض المعلومات أو الدقة مقابل الكفاءة. Flash Attention، في المقابل، **لا يُغيِّر الصيغة الرياضية إطلاقًا** — فهو يحسب بالضبط نفس $\text{softmax}(QK^T/\sqrt{d_k})V$ القياسي، لكنه يُعيد تنظيم **ترتيب وتوقيت العمليات الحسابية** بذكاء لتقليل عدد مرات نقل البيانات بين ذاكرة GPU السريعة الصغيرة (SRAM) والذاكرة الأبطأ الكبيرة (HBM)، باستخدام تقنيات مثل التقسيم لكتل (tiling) وخوارزمية online softmax التي تُحدِّث مجموع softmax تدريجيًا دون الحاجة لرؤية صف المصفوفة كاملًا دفعة واحدة. النتيجة النهائية مطابقة رياضيًا (في حدود الدقة العددية الطبيعية) للتنفيذ القياسي، لكنها أسرع بكثير وتستهلك ذاكرة أقل بكثير — هذا هو الفرق الجوهري: تحسين **التنفيذ** مقابل تغيير **الخوارزمية** نفسها.

---


## 10. تمرين عملي

1. نفّذ `RotaryPositionalEmbedding` وطبّقها على متجهين عشوائيين عند موضعين مختلفين (مثلًا 5 و10)، وتحقق عدديًا أن حاصل ضربهما الداخلي يساوي تقريبًا حاصل ضرب نفس المتجهين عند موضعين آخرين بنفس الفرق (مثلًا 100 و105).
2. احسب `compute_kv_cache_size` لنموذج بمواصفاتك المفضلة بثلاث قيم مختلفة لعدد مجموعات GQA (1، 8، وعدد الرؤوس الكامل)، وارسم مخططًا بيانيًا يوضح العلاقة.
3. شغّل `compare_standard_vs_flash` (أو ما يعادلها) على بيئتك، وسجّل نسبة التسريع والتوفير الذاكري الفعليين.
4. اقرأ عن تطبيق ALiBi وقارنه ذهنيًا (دون كود بالضرورة) مع RoPE من حيث البساطة والاستقراء — أيهما تختار لنموذج يحتاج التعميم لسياقات أطول بـ10 أضعاف من بيانات التدريب؟

---


## 11. مراجع للتعمّق

- Su, J., et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding* (RoPE).
- Press, O., et al. (2021). *Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation* (ALiBi).
- Ainslie, J., et al. (2023). *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints*.
- Dao, T., et al. (2022). *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*.
- Liu, H., et al. (2023). *Ring Attention with Blockwise Transformers for Near-Infinite Context*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المعماريات الحديثة / الفصل 14: الترميز الموضعي المتقدم وآليات الانتباه الفعّالة.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
